In [ ]:
# Animate x-v_x phase space from PIC_Part_*.csv snapshots.
# Saves xvx_animation.mp4 (falls back to GIF if ffmpeg is unavailable).

import glob
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import Video, display
from tqdm import tqdm

PATTERN = "PIC_Part_*.csv"
OUT_MP4 = "xvx_animation.mp4"
OUT_GIF = "xvx_animation.gif"
FPS = 10
SCATTER_SIZE = 8
SCATTER_ALPHA = 0.7
# If True, divide momentum (Field_2_0) by estimated mass per frame (val_init style).
USE_VELOCITY = False


def extract_step(path: Path) -> int:
    m = re.search(r"PIC_Part_(\d+)\.csv$", path.name)
    return int(m.group(1)) if m else -1


files = sorted(glob.glob(PATTERN), key=lambda f: extract_step(Path(f)))
if not files:
    raise FileNotFoundError(f"No files matching pattern: {PATTERN}")

required = {"X", "Y", "Field_2_0"}
frames = []
for f in tqdm(files, desc="Reading CSV frames"):
    df = pd.read_csv(f, usecols=["X", "Y", "Field_2_0"])
    if not required.issubset(df.columns):
        raise ValueError(f"{f} missing columns. Found: {list(df.columns)}")
    frames.append(df)

all_x = []
all_vx = []
for df in frames:
    x = df["X"].to_numpy()
    y = df["Y"].to_numpy()
    vx = df["Field_2_0"].to_numpy()
    if USE_VELOCITY:
        m = (x.max() - x.min()) * (y.max() - y.min()) / len(x)
        vx = vx / m
    all_x.append(x)
    all_vx.append(vx)

all_x = np.concatenate(all_x)
all_vx = np.concatenate(all_vx)

pad_x = (all_x.max() - all_x.min()) * 0.02 if np.ptp(all_x) != 0 else 1.0
pad_v = (all_vx.max() - all_vx.min()) * 0.02 if np.ptp(all_vx) != 0 else 1.0
xlim = (all_x.min() - pad_x, all_x.max() + pad_x)
vxlim = (all_vx.min() - pad_v, all_vx.max() + pad_v)

ylabel = "v_x" if USE_VELOCITY else "p_x / m (Field_2_0)"

fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter([], [], s=SCATTER_SIZE, alpha=SCATTER_ALPHA, c="#1f77b4", edgecolors="none")
ax.set_xlim(*xlim)
ax.set_ylim(*vxlim)
ax.set_xlabel("x")
ax.set_ylabel(ylabel)
title = ax.set_title("")
ax.grid(True, alpha=0.3)


def init():
    sc.set_offsets(np.empty((0, 2)))
    title.set_text("")
    return sc, title


pbar = tqdm(total=len(frames), desc="Rendering frames", leave=False)


def update(i):
    df = frames[i]
    x = df["X"].to_numpy()
    y = df["Y"].to_numpy()
    vx = df["Field_2_0"].to_numpy()
    if USE_VELOCITY:
        m = (x.max() - x.min()) * (y.max() - y.min()) / len(x)
        vx = vx / m
    sc.set_offsets(np.column_stack([x, vx]))
    step = extract_step(Path(files[i]))
    title.set_text(f"x-v_x  |  step {step}  |  frame {i + 1}/{len(frames)}")
    pbar.update(1)
    return sc, title


anim = animation.FuncAnimation(
    fig,
    update,
    frames=len(frames),
    init_func=init,
    blit=True,
    interval=100,
    repeat=False,
)

saved_path = None
try:
    writer = animation.FFMpegWriter(
        fps=FPS, metadata=dict(artist="generated"), bitrate=2000
    )
    anim.save(OUT_MP4, writer=writer, dpi=150)
    saved_path = OUT_MP4
except Exception:
    try:
        anim.save(OUT_GIF, writer="pillow", fps=FPS, dpi=150)
        saved_path = OUT_GIF
    except Exception as e:
        pbar.close()
        plt.close(fig)
        raise RuntimeError(
            "Failed to save animation as MP4 or GIF. Ensure ffmpeg or pillow is available."
        ) from e
finally:
    pbar.close()

plt.close(fig)
display(Video(saved_path, embed=True))
print(f"Saved animation to: {saved_path}")